In [11]:
import chromadb
from chromadb.config import Settings

In [2]:
# creating client
client = chromadb.PersistentClient(path="../chromadb")

In [3]:
# creating collection
collection = client.create_collection(name="app")

In [4]:
# adding data to collection 
collection.add(
    ids=["1"],
    embeddings=[[1,3,4,5]],
    documents=["hello world"]
)

In [6]:
# getting a document
collection.get(ids="1")

{'ids': ['1'],
 'embeddings': None,
 'documents': ['hello world'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None]}

In [7]:
collection.delete(ids="1")

{'deleted': 1}

#### Configuring ChromaDB

In [13]:
path = "../chromadb/apple"
tenant = "apple"
database = "apple_db"

settings = Settings(
    is_persistent=True,
    persist_directory=path
)

admin = chromadb.AdminClient(
    settings=settings
)

# creating tanent
admin.create_tenant(name=tenant)
admin.create_database(tenant=tenant,name=database)

In [14]:
# initilizing the database by creating client
client = chromadb.PersistentClient(
    path=path,
    tenant=tenant,
    database=database
)

In [17]:
# creating the collection
vector_collection = client.create_collection(name="vector_cl")

In [20]:
# adding documents
display(vector_collection.tenant)
display(vector_collection.database)

'apple'

'apple_db'

In [9]:
"""
Implement the vector store using chroma db
"""

import uuid
import chromadb
from chromadb.config import Settings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


# function to read the pdf file from the particular file path and return the document
def data_ingestion(file_path):
    try:
        loader = PyMuPDFLoader(file_path=file_path)
        print(loader.load())
        return loader.load()
    except Exception as e:
        print(e)


In [ ]:

# class design to convert the text into vectors using open source model [all-MiniLM-L6-v2]
# chunking the document with the size of 600 and overlap size is 200
class Embdedding:
    def __init__(self, model, chunk_size=600, chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model)

    def chunk_documents(self, documents):
        try:
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap,
                separators=["\n\n", "\n", " ", ""],
            )
            chunks = splitter.split_documents(documents=documents)
            
            return chunks
        except Exception as e:
            print(e)

    def embed_chunks(self, chunks):
        try:
            text = [chunk.page_content for chunk in chunks]
            vectors = self.model.encode(text)
            return vectors
        except Exception as e:
            print(e)

In [10]:

path = "../data/version_4.pdf"
model = "all-MiniLM-L6-v2"
presistent_dir = "./day_5/database/chromadb"
tenant = "resume"
database = "resume_db"

document = data_ingestion(path)
print(document)

[Document(metadata={'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/version_4.pdf', 'file_path': '../data/version_4.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'BHASKAR resume', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='BHASKAR RAI \nDATA SCIENCE INTERN | AI/ML INTERN \nMechinagar-3 Jhapa | iamraibhaskar@gmail.com | 9815048958/9762902543 | Github | \nLinkedin \n________________________________________________________________ \nPROFILE \nData Science and Machine Learning enthusiast with hands-on experience in Python, SQL, \nPandas, NumPy, Scikit-learn, and exploratory data analysis. Experienced in developing \nmachine learning models, analyzing datasets, and extracting insights \nthrough data visualization. Strong foundation in statistics, linear algebra, NLP, and deep \nlearning, with a focus on applying machine learning techniq

#### Chunks


In [11]:
embedding =Embdedding(model=model)
chunks = embedding.chunk_documents(documents=document)
print(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4715.22it/s]


[Document(metadata={'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/version_4.pdf', 'file_path': '../data/version_4.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'BHASKAR resume', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='BHASKAR RAI \nDATA SCIENCE INTERN | AI/ML INTERN \nMechinagar-3 Jhapa | iamraibhaskar@gmail.com | 9815048958/9762902543 | Github | \nLinkedin \n________________________________________________________________ \nPROFILE \nData Science and Machine Learning enthusiast with hands-on experience in Python, SQL, \nPandas, NumPy, Scikit-learn, and exploratory data analysis. Experienced in developing \nmachine learning models, analyzing datasets, and extracting insights \nthrough data visualization. Strong foundation in statistics, linear algebra, NLP, and deep'), Document(metadata={'producer': 'Skia/PDF m154 Google Docs 

In [13]:
chunks[0].page_content

'BHASKAR RAI \nDATA SCIENCE INTERN | AI/ML INTERN \nMechinagar-3 Jhapa | iamraibhaskar@gmail.com | 9815048958/9762902543 | Github | \nLinkedin \n________________________________________________________________ \nPROFILE \nData Science and Machine Learning enthusiast with hands-on experience in Python, SQL, \nPandas, NumPy, Scikit-learn, and exploratory data analysis. Experienced in developing \nmachine learning models, analyzing datasets, and extracting insights \nthrough data visualization. Strong foundation in statistics, linear algebra, NLP, and deep'

#### Convert into vectors

In [14]:
vectors = embedding.embed_chunks(chunks)
print(vectors)

[[-4.04746793e-02 -7.06125274e-02  3.51351574e-02 ... -1.34270946e-02
  -9.14014131e-02 -6.67194035e-05]
 [-1.72172152e-02 -3.05712707e-02 -1.43890865e-02 ... -5.58221014e-04
  -5.79467788e-02  3.26732993e-02]
 [-9.62038804e-03  9.33902711e-03 -3.89824249e-02 ...  8.46588984e-02
  -2.83441786e-02  3.34684178e-02]
 [ 5.17792366e-02  2.78352592e-02 -7.32968077e-02 ...  6.98092282e-02
  -8.37842934e-03  4.95320484e-02]
 [ 1.15655577e-02 -3.46318930e-02 -7.92218074e-02 ...  5.30093648e-02
   7.05030514e-03  9.99882072e-02]
 [-2.96451896e-02 -4.44564894e-02 -3.90852941e-03 ... -9.27430484e-03
  -5.98287843e-02  2.37763524e-02]]
